# AI Model Benchmark Suite — llama-cpp-python Backend

Evaluates a set of local GGUF models across **10 factual tasks** and **10 tool-calling checks**.

For each model the pipeline is:
1. **Load** the GGUF file into llama-cpp-python (`Llama` object)
2. **Run** the factual benchmark + tool-calling tests
3. **Unload** (delete the `Llama` object and force GC) before the next model

Metrics reported:
| Metric | Description |
|---|---|
| **Throughput** | Output tokens / second |
| **Accuracy** | Factual tasks judged correct by the evaluator LLM |
| **Latency** | Wall-clock seconds per task |
| **Conciseness** | Average response length in characters |
| **Tool score** | How many of the 10 tool calls returned valid results |

## 1 · Install dependencies

In [1]:
# CPU-only build (default). For CUDA add: CMAKE_ARGS='-DGGML_CUDA=on'
!uv add pandas matplotlib numpy pydantic llama-cpp-python

# Windows install (CUDA):
# !$env:CMAKE_ARGS="-DGGML_CUDA=on"
# !uv add llama-cpp-python


Resolved 250 packages in 1ms
Checked 240 packages in 16ms


In [2]:
# Windows install
# !$env:CMAKE_ARGS="-DGGML_CUDA=on"
# !uv add llama-cpp-python

## 2 · Imports

In [3]:
"""
Standard-library and third-party imports.

llama_cpp   – high-level Llama class (model load / unload / inference).
pydantic    – BaseModel used for typed tool schemas and tool-result wrappers.
asyncio     – drives async helpers (kept async so tool dispatch can stay
              consistent with any future async llama-cpp wrappers).
dataclass   – lightweight value objects for tasks and results.
time        – high-resolution wall-clock timing via perf_counter.
gc          – explicit garbage collection to free VRAM after each model.
pandas / matplotlib / numpy – tabular aggregation and visualisation.
"""

import gc
import json
import time
import llama_cpp

from dataclasses import dataclass
from datetime import date
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

from llama_cpp import Llama


## 3 · Configuration constants

In [4]:
"""
Global configuration.

BENCHMARK_MODELS  – ordered dict of  display_name → local .gguf path.
                    Edit the paths to match your filesystem.
EVALUATOR_MODEL   – display name of the model used to judge factual answers;
                    must also be present in BENCHMARK_MODELS.
N_GPU_LAYERS      – number of transformer layers to offload to the GPU.
                    -1 = all layers (full GPU), 0 = CPU only.
N_CTX             – context-window size used for every model.

MODEL_CHAT_FORMATS – maps each model display name to the chat_format string
                     that llama-cpp-python needs to activate tool-calling.
                     Without this, the Llama constructor defaults to an
                     autodetected format that often does NOT wire up tool
                     routing, so create_chat_completion(tools=...) silently
                     produces text answers instead of tool_calls.

Supported chat_format values for tool calling in llama-cpp-python:
  "chatml-function-calling"  – ChatML wrapper w/ JSON tool schema injection
                               works for Qwen2.5, Phi-4-mini, many others
  "functionary-v1/v2/v3"    – Meetkai Functionary models only
  "llama-3"                  – Llama 3.x native format (JSON in <|python_tag|>)
  None / omit key           – llama-cpp-python autodetects from GGUF metadata;
                               safe for factual tasks but unreliable for tools
"""

BENCHMARK_MODELS: dict[str, str] = {
    # "Qwen3-0.6B":    "./models/qwen3-0.6b-q4_k_m.gguf",
    # "LFM2.5-1.2B":   "./models/LFM2.5-1.2B-Instruct-Q4_0.gguf",
    # "LFM2.5-Think":  "./models/LFM2.5-1.2B-Thinking-Q4_0.gguf",
    # "LFM2-Tool":  "./models/LFM2-1.2B-Tool-Q4_0.gguf",
    # "LFM2-8B-A1B":   "./models/LFM2.5-8B-A1B-Q4_0.gguf",
    "Phi4-mini":     "./models/phi-4-mini-instruct-q4_0.gguf",
}

# BENCHMARK_MODELS: dict[str, str] = {
#     "LFM2.5-1.2B":   "/media/eduardo/TOSHIBA/data/LFM2.5-1.2B-Instruct-Q4_0.gguf",
#     "LFM2-8B-A1B":   "/media/eduardo/TOSHIBA/data/LFM2.5-8B-A1B-Q4_0.gguf",
#     "LFM2.5-Think":  "/media/eduardo/TOSHIBA/data/LFM2.5-1.2B-Thinking-Q4_0.gguf",
#     "LFM2-Tool":  "/media/eduardo/TOSHIBA/data/LFM2-1.2B-Tool-Q4_0.gguf",
#     "Qwen3-0.6B":    "/media/eduardo/TOSHIBA/data/Qwen3-0.6B-Q8_0.gguf",
#     "Phi4-mini":     "/media/eduardo/TOSHIBA/data/phi-4-mini-instruct-q4_0.gguf",
# }

EVALUATOR_MODEL: str = "LFM2.5-1.2B"   # must be a key in BENCHMARK_MODELS

N_GPU_LAYERS: int = -1    # -1 = all layers on GPU; 0 = CPU only
N_CTX:        int = 4084   

# ── Per-model chat_format for tool calling ───────────────────────────────────
# llama-cpp-python must know the chat format at Llama() construction time to
# correctly inject tool schemas into the prompt and parse tool_calls out of
# the response.  If the format is wrong or missing, tool_calls is always [].
#
# Key findings per model family:
#   LFM2.5 / LFM2  – use custom <|tool_call_start|>...<|tool_call_end|> tokens;
#                     llama-cpp-python has no native handler for this format.
#                     "chatml-function-calling" is the closest working option
#                     but results vary; these models may not emit tool_calls
#                     through the llama-cpp-python API at all.
#   Qwen2.5 / Qwen3 – "chatml-function-calling" (Hermes-2-Pro JSON format)
#                     is natively supported and reliably emits tool_calls.
#   Phi-4-mini      – uses a ChatML-derived template; "chatml-function-calling"
#                     works. phi-4 is listed as "Generic" in the llama.cpp
#                     function-calling matrix, meaning it falls back to generic
#                     tool schema injection.
#
# Set a key to None to skip chat_format override (autodetect from GGUF).
MODEL_CHAT_FORMATS: dict[str, str | None] = {
    "LFM2.5-1.2B":   "chatml-function-calling",
    "LFM2.5-Think":  "chatml-function-calling",
    "Qwen3-0.6B":    "chatml-function-calling",
    "LFM2-Tool":     "chatml-function-calling",
    "LFM2-8B-A1B":   "chatml-function-calling",
    "Phi4-mini":     "chatml-function-calling",
}

print("Benchmark models:")
for name, path in BENCHMARK_MODELS.items():
    fmt = MODEL_CHAT_FORMATS.get(name, "autodetect")
    tag = " ← evaluator" if name == EVALUATOR_MODEL else ""
    print(f"  {name:<20} chat_format={fmt!r:<28} {path}{tag}")


Benchmark models:
  Phi4-mini            chat_format='chatml-function-calling'    ./models/phi-4-mini-instruct-q4_0.gguf


## 4 · Data models

In [5]:
"""
Immutable value objects used throughout the benchmark pipeline.

BenchmarkTask  – a prompt paired with its canonical reference answer.
TaskResult     – full record produced after one model runs one task,
                 including timing, token counts, and the evaluator verdict.
ToolResult     – outcome of a single tool-call test (one of the 3 tools).
ModelToolScore – aggregated tool-calling results for one model.
"""


@dataclass(frozen=True)
class BenchmarkTask:
    """A single evaluation task with a prompt and a reference answer.

    Attributes:
        task_id:         1-based sequential identifier.
        prompt:          Natural-language question sent to each model.
        expected_answer: Canonical answer used by the evaluator LLM.
    """
    task_id: int
    prompt: str
    expected_answer: str


@dataclass
class TaskResult:
    """Full record produced after running one model on one task.

    Attributes:
        model:             Display name of the model.
        task_id:           Matches BenchmarkTask.task_id.
        prompt:            Original question.
        expected_answer:   Canonical answer.
        model_answer:      Raw text output from the model.
        elapsed_seconds:   Wall-clock duration of the completion call.
        input_tokens:      Tokens consumed by the prompt.
        output_tokens:     Tokens generated in the response.
        tokens_per_second: output_tokens / elapsed_seconds.
        response_length:   Character count of model_answer.
        is_correct:        Evaluator verdict; None until evaluation completes.
        error:             Exception message if the run failed, else None.
    """
    model: str
    task_id: int
    prompt: str
    expected_answer: str
    model_answer: str
    elapsed_seconds: float
    input_tokens: int
    output_tokens: int
    tokens_per_second: float
    response_length: int
    is_correct: Optional[bool] = None
    error: Optional[str] = None


@dataclass
class ToolResult:
    """Outcome of one tool-call test against a single model.

    Attributes:
        tool_name:  Name of the tool that was invoked.
        called:     Whether the model actually emitted a tool call.
        valid:      Whether the tool result parsed correctly as the expected type.
        raw:        The raw string the model returned (or error message).
    """
    tool_name: str
    called: bool
    valid: bool
    raw: str
    elapsed_seconds: float = 0.0


@dataclass
class ModelToolScore:
    """Aggregated tool-call results for one model.

    Attributes:
        model:   Display name of the model.
        results: List of ToolResult, one per tool.
        score:   Number of tools that were both called and returned valid data.
    """
    model: str
    results: list[ToolResult]

    @property
    def score(self) -> int:
        return sum(1 for r in self.results if r.called and r.valid)


## 5 · Pydantic tool schemas & tool registry

In [6]:
"""
Ten static tools exposed to every model during the tool-calling benchmark.

Tool  1 – get_current_date     Returns today's date (YYYY-MM-DD).
Tool  2 – get_tasks            Returns a static to-do list.
Tool  3 – list_files           Returns a static directory listing.
Tool  4 – get_weather          Returns mocked current weather for a city.
Tool  5 – calculator           Evaluates a basic arithmetic expression.
Tool  6 – get_user_profile     Returns a static logged-in user profile.
Tool  7 – search_knowledge_base Searches a mock internal KB by keyword.
Tool  8 – create_reminder      Schedules a reminder (static echo-back).
Tool  9 – get_stock_price      Returns a mocked stock price for a ticker.
Tool 10 – translate_text       Returns a mocked translation of a snippet.

Each tool is defined as:
  • A Pydantic BaseModel subclass for the *parameters* schema (used to
    generate the OpenAI-compatible JSON Schema sent to the model).
  • A plain Python function that executes the tool and returns a dict.

The TOOL_DEFINITIONS list is the OpenAI-format tool spec fed to
create_chat_completion(tools=...).  TOOL_HANDLERS maps tool name → callable.

Docstring / description quality matters: the model reads the "description"
field of each tool definition to decide *when* to call it.  Vague or
overlapping descriptions cause the model to pick the wrong tool or none at
all.  Every description here follows three rules:
  1. State what it returns, not just what it "does".
  2. List what it does NOT do, to prevent confusion with similar tools.
  3. Name the required argument (if any) so the model fills it correctly.
"""


# ── Schema models (parameters) ───────────────────────────────────────────────

class GetCurrentDateParams(BaseModel):
    """No parameters — the tool always returns today's calendar date."""
    pass


class GetTasksParams(BaseModel):
    """No parameters — returns the full static to-do / task list."""
    pass


class ListFilesParams(BaseModel):
    """Optional directory path; the static implementation always returns the
    same file list regardless of the value supplied."""
    directory: str = Field(
        default=".",
        description=(
            "Filesystem path of the directory to list.  "
            "Use '.' for the current working directory."
        ),
    )


class GetWeatherParams(BaseModel):
    """City name for which to return current weather conditions."""
    city: str = Field(
        description=(
            "Name of the city, e.g. 'London' or 'New York'.  "
            "Do not include country codes or coordinates."
        ),
    )


class CalculatorParams(BaseModel):
    """A single arithmetic expression to evaluate."""
    expression: str = Field(
        description=(
            "A pure arithmetic expression using +, -, *, / and parentheses, "
            "e.g. '(3 + 5) * 2'.  Do not include variable names or functions."
        ),
    )


class GetUserProfileParams(BaseModel):
    """No parameters — returns the profile of the currently logged-in user."""
    pass


class SearchKnowledgeBaseParams(BaseModel):
    """Keyword or phrase to search for in the internal knowledge base."""
    query: str = Field(
        description=(
            "One or more keywords describing the topic to look up, "
            "e.g. 'vacation policy' or 'VPN setup'.  "
            "Returns the single most-relevant article title and summary."
        ),
    )


class CreateReminderParams(BaseModel):
    """Text and due-date for a new reminder."""
    text: str = Field(
        description="The reminder message, e.g. 'Call dentist'.",
    )
    due: str = Field(
        description=(
            "Due date/time in ISO-8601 format, e.g. '2025-08-01T09:00:00'.  "
            "Date-only values like '2025-08-01' are also accepted."
        ),
    )


class GetStockPriceParams(BaseModel):
    """Ticker symbol for which to return the current mocked stock price."""
    ticker: str = Field(
        description=(
            "Exchange ticker symbol in upper-case, e.g. 'AAPL', 'TSLA', 'MSFT'.  "
            "Do not include the exchange prefix."
        ),
    )


class TranslateTextParams(BaseModel):
    """Text to translate and the target language."""
    text: str = Field(
        description="The source text to translate.",
    )
    target_language: str = Field(
        description=(
            "Full English name of the target language, e.g. 'French', 'Spanish', "
            "'German', 'Japanese'.  Do not use ISO codes."
        ),
    )


# ── Return-value models (used for validation) ────────────────────────────────

class CurrentDateResult(BaseModel):
    date: str = Field(description="Today's date in YYYY-MM-DD format.")


class TasksResult(BaseModel):
    tasks: list[str] = Field(description="List of pending task strings.")


class FilesResult(BaseModel):
    files: list[str] = Field(description="List of file names in the directory.")


class WeatherResult(BaseModel):
    city: str         = Field(description="City name as supplied.")
    condition: str    = Field(description="Sky condition, e.g. 'Sunny'.")
    temperature_c: float = Field(description="Temperature in degrees Celsius.")


class CalculatorResult(BaseModel):
    expression: str = Field(description="The original expression.")
    result: float   = Field(description="Numeric result of the expression.")


class UserProfileResult(BaseModel):
    username: str  = Field(description="Login username.")
    full_name: str = Field(description="Display name.")
    email: str     = Field(description="Primary e-mail address.")
    role: str      = Field(description="User role, e.g. 'admin' or 'viewer'.")


class KnowledgeBaseResult(BaseModel):
    query: str   = Field(description="The search query that was submitted.")
    title: str   = Field(description="Title of the best-matching article.")
    summary: str = Field(description="One-sentence summary of the article.")


class ReminderResult(BaseModel):
    id: str   = Field(description="Auto-generated reminder ID.")
    text: str = Field(description="Reminder message as stored.")
    due: str  = Field(description="Due date/time as supplied.")


class StockPriceResult(BaseModel):
    ticker: str   = Field(description="Ticker symbol as supplied.")
    price: float  = Field(description="Mocked current price in USD.")
    currency: str = Field(description="Currency of the price.")


class TranslationResult(BaseModel):
    source_text:     str = Field(description="The original text.")
    target_language: str = Field(description="Target language as supplied.")
    translated_text: str = Field(description="Mocked translated output.")


# ── Tool implementations ─────────────────────────────────────────────────────

def _tool_get_current_date(_args: dict) -> dict:
    return {"date": date.today().isoformat()}


def _tool_get_tasks(_args: dict) -> dict:
    return {"tasks": ["read book", "buy groceries", "call dentist", "review PR #42"]}


def _tool_list_files(_args: dict) -> dict:
    return {
        "files": [
            "report_q1.pdf",
            "budget_2025.xlsx",
            "meeting_notes.md",
            "logo_final.png",
            "deployment_script.sh",
        ]
    }


def _tool_get_weather(args: dict) -> dict:
    city = args.get("city", "Unknown")
    return {"city": city, "condition": "Partly cloudy", "temperature_c": 18.5}


def _tool_calculator(args: dict) -> dict:
    expr = args.get("expression", "")
    try:
        # Restrict to safe arithmetic only — no builtins, no names
        result = float(eval(expr, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception:
        result = float("nan")
    return {"expression": expr, "result": result}


def _tool_get_user_profile(args: dict) -> dict:
    return {
        "username":  "jdoe",
        "full_name": "Jane Doe",
        "email":     "jdoe@example.com",
        "role":      "admin",
    }


def _tool_search_knowledge_base(args: dict) -> dict:
    query = args.get("query", "")
    return {
        "query":   query,
        "title":   "Getting Started with the Internal Wiki",
        "summary": "Overview of how to search, edit, and create articles in the company knowledge base.",
    }


def _tool_create_reminder(args: dict) -> dict:
    return {
        "id":   "rem-0042",
        "text": args.get("text", ""),
        "due":  args.get("due", ""),
    }


def _tool_get_stock_price(args: dict) -> dict:
    ticker = args.get("ticker", "").upper()
    prices = {"AAPL": 213.49, "TSLA": 248.10, "MSFT": 431.87, "GOOG": 178.02}
    return {
        "ticker":   ticker,
        "price":    prices.get(ticker, 99.99),
        "currency": "USD",
    }


def _tool_translate_text(args: dict) -> dict:
    return {
        "source_text":     args.get("text", ""),
        "target_language": args.get("target_language", ""),
        "translated_text": f"[{args.get('target_language', 'Unknown')} translation of: {args.get('text', '')}]",
    }


# ── Validator map (tool name → Pydantic result model) ────────────────────────

TOOL_RESULT_VALIDATORS: dict[str, type[BaseModel]] = {
    "get_current_date":      CurrentDateResult,
    "get_tasks":             TasksResult,
    "list_files":            FilesResult,
    "get_weather":           WeatherResult,
    "calculator":            CalculatorResult,
    "get_user_profile":      UserProfileResult,
    "search_knowledge_base": KnowledgeBaseResult,
    "create_reminder":       ReminderResult,
    "get_stock_price":       StockPriceResult,
    "translate_text":        TranslationResult,
}

TOOL_HANDLERS: dict[str, callable] = {
    "get_current_date":      _tool_get_current_date,
    "get_tasks":             _tool_get_tasks,
    "list_files":            _tool_list_files,
    "get_weather":           _tool_get_weather,
    "calculator":            _tool_calculator,
    "get_user_profile":      _tool_get_user_profile,
    "search_knowledge_base": _tool_search_knowledge_base,
    "create_reminder":       _tool_create_reminder,
    "get_stock_price":       _tool_get_stock_price,
    "translate_text":        _tool_translate_text,
}


def _schema(model: type[BaseModel]) -> dict:
    """Return a JSON-Schema dict from a Pydantic model, suitable for tool defs."""
    return model.model_json_schema()


TOOL_DEFINITIONS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": (
                "Returns today's calendar date as a string in YYYY-MM-DD format.  "
                "Use this when the user asks what day or date it is.  "
                "Does NOT return the current time."
            ),
            "parameters": _schema(GetCurrentDateParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_tasks",
            "description": (
                "Returns the user's full list of pending to-do items.  "
                "Use this when the user asks about their tasks, to-dos, or what they need to do.  "
                "Returns all tasks; does not support filtering."
            ),
            "parameters": _schema(GetTasksParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": (
                "Lists the files present in a given directory.  "
                "Use this when the user asks what files exist or wants a directory listing.  "
                "Returns file names only, not file contents or sizes."
            ),
            "parameters": _schema(ListFilesParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Returns the current weather conditions (sky condition and temperature in °C) "
                "for a specified city.  "
                "Use this when the user asks about the weather or temperature in a location.  "
                "Requires the city name; does not accept coordinates."
            ),
            "parameters": _schema(GetWeatherParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluates a basic arithmetic expression and returns the numeric result.  "
                "Use this for any calculation involving +, -, *, / or parentheses.  "
                "Does NOT support trigonometry, logarithms, or variable names."
            ),
            "parameters": _schema(CalculatorParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_profile",
            "description": (
                "Returns the profile of the currently authenticated user, including "
                "their username, full name, email address, and role.  "
                "Use this when the user asks who they are logged in as or about their account."
            ),
            "parameters": _schema(GetUserProfileParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": (
                "Searches the internal company knowledge base for articles matching a keyword "
                "or phrase and returns the title and summary of the best match.  "
                "Use this when the user asks a question that might be answered by internal documentation.  "
                "Requires a search query string."
            ),
            "parameters": _schema(SearchKnowledgeBaseParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_reminder",
            "description": (
                "Creates a new reminder with a message and a due date/time, "
                "and returns the saved reminder ID.  "
                "Use this when the user asks to set, schedule, or add a reminder.  "
                "Requires both the reminder text and a due date."
            ),
            "parameters": _schema(CreateReminderParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": (
                "Returns the current (mocked) stock price in USD for a given ticker symbol.  "
                "Use this when the user asks about a stock price or share value.  "
                "Requires the ticker symbol in upper-case, e.g. 'AAPL'."
            ),
            "parameters": _schema(GetStockPriceParams),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "translate_text",
            "description": (
                "Translates a piece of text into a specified target language and returns "
                "the translated string.  "
                "Use this when the user asks to translate something.  "
                "Requires both the source text and the full English name of the target language."
            ),
            "parameters": _schema(TranslateTextParams),
        },
    },
]

print(f"Registered {len(TOOL_DEFINITIONS)} tools:")
for t in TOOL_DEFINITIONS:
    print(f"  • {t['function']['name']:<24} — {t['function']['description'][:70]}")

Registered 10 tools:
  • get_current_date         — Returns today's calendar date as a string in YYYY-MM-DD format.  Use t
  • get_tasks                — Returns the user's full list of pending to-do items.  Use this when th
  • list_files               — Lists the files present in a given directory.  Use this when the user 
  • get_weather              — Returns the current weather conditions (sky condition and temperature 
  • calculator               — Evaluates a basic arithmetic expression and returns the numeric result
  • get_user_profile         — Returns the profile of the currently authenticated user, including the
  • search_knowledge_base    — Searches the internal company knowledge base for articles matching a k
  • create_reminder          — Creates a new reminder with a message and a due date/time, and returns
  • get_stock_price          — Returns the current (mocked) stock price in USD for a given ticker sym
  • translate_text           — Translates a piece of text int

## 6 · Benchmark task definitions

In [7]:
"""
Ten factual tasks covering arithmetic, geography, history, science, and
language.  Each expected_answer is a short, unambiguous canonical string
that the evaluator LLM compares against the model's free-form response.
"""

TASKS: list[BenchmarkTask] = [
    BenchmarkTask(1,  "What is the capital of France?",                                  "Paris"),
    BenchmarkTask(2,  "What is 2 raised to the power of 10?",                            "1024"),
    BenchmarkTask(3,  "What is the chemical symbol for gold?",                           "Au"),
    BenchmarkTask(4,  "Who wrote Romeo and Juliet?",                                     "William Shakespeare"),
    BenchmarkTask(5,  "What planet is closest to the Sun?",                              "Mercury"),
    BenchmarkTask(6,  "In what year did World War II end?",                              "1945"),
    BenchmarkTask(7,  "What is the square root of 144?",                                 "12"),
    BenchmarkTask(8,  "How many sides does a hexagon have?",                             "6"),
    BenchmarkTask(9,  "What is the boiling point of water in Celsius at sea level?",    "100"),
    BenchmarkTask(10, "What programming language is named after a British comedy group?", "Python"),
]

print(f"Loaded {len(TASKS)} benchmark tasks.")
for t in TASKS:
    print(f"  [{t.task_id:02d}] {t.prompt}  →  {t.expected_answer}")

Loaded 10 benchmark tasks.
  [01] What is the capital of France?  →  Paris
  [02] What is 2 raised to the power of 10?  →  1024
  [03] What is the chemical symbol for gold?  →  Au
  [04] Who wrote Romeo and Juliet?  →  William Shakespeare
  [05] What planet is closest to the Sun?  →  Mercury
  [06] In what year did World War II end?  →  1945
  [07] What is the square root of 144?  →  12
  [08] How many sides does a hexagon have?  →  6
  [09] What is the boiling point of water in Celsius at sea level?  →  100
  [10] What programming language is named after a British comedy group?  →  Python


## 7 · Model loader & unloader

In [8]:
"""
load_model()   – creates a Llama instance from a local .gguf path.
unload_model() – deletes the instance and runs the garbage collector so
                 GPU/CPU memory is released before the next model loads.

CRITICAL: chat_format must be passed to the Llama() constructor – it cannot
be set later.  llama-cpp-python uses it at construction time to select the
LlamaChatCompletionHandler that will be called by create_chat_completion().
Without the right handler selected, passing tools= to create_chat_completion
has no effect: the handler does not inject the JSON schemas into the prompt
and cannot parse <tool_call> tokens out of the response, so tool_calls is
always an empty list.
"""


def load_model(model_path: str, chat_format: str | None = None,
               verbose: bool = False) -> Llama:
    """Load a GGUF model from disk into llama-cpp-python.

    Args:
        model_path:  Absolute or relative path to the .gguf file.
        chat_format: Chat format string passed to the Llama constructor.
                     Controls which completion handler is used, including
                     whether tool calling is active.  None = autodetect.
        verbose:     If True, llama.cpp prints its own load diagnostics.

    Returns:
        A ready-to-use :class:`llama_cpp.Llama` instance.
    """
    print(f"  Loading {model_path} …", end=" ", flush=True)
    t0 = time.perf_counter()
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=N_GPU_LAYERS,
        n_ctx=N_CTX,
        chat_format=chat_format,   # None = let llama-cpp autodetect
        verbose=verbose,
        last_n_tokens_size=0, 
    )
    llm.set_cache(None)
    elapsed = time.perf_counter() - t0
    fmt_tag = f" [{chat_format}]" if chat_format else ""
    print(f"done ({elapsed:.1f}s){fmt_tag}")
    return llm


def unload_model(llm: Llama) -> None:
    try:
        llm.close()
    except Exception:
        pass
    try:
        llm._model.__del__()  # force C-level free
    except Exception:
        pass
    del llm
    for _ in range(3):
        gc.collect()
    time.sleep(2)  # give the driver time to reclaim pages
    print("  Model unloaded.")

    
def clear_context(llm):
    """Nuke every piece of inference state between calls."""
    # 1. KV cache — C-level API first, then Python wrapper
    cleared = False
    try:
        llama_cpp.llama_kv_cache_clear(llm._ctx.ctx)
        cleared = True
    except Exception:
        pass
    if not cleared:
        try:
            llm._ctx.kv_cache_clear()
        except Exception:
            pass

    # 2. Token position counters — if nonzero, next eval appends instead of starting fresh
    for attr in ("n_tokens", "_n_tokens"):
        if hasattr(llm, attr):
            try:
                setattr(llm, attr, 0)
            except Exception:
                pass

    # 3. Score and input-id buffers
    try:
        llm.input_ids[:] = 0
    except Exception:
        pass
    try:
        llm.scores[:] = 0
    except Exception:
        pass

    # 4. Reset high-level state
    if hasattr(llm, "reset"):
        try:
            llm.reset()
        except Exception:
            pass

    # 5. Sampler chain
    if getattr(llm, "_sampler", None) is not None:
        try:
            llm._sampler.reset()
        except Exception:
            pass

## 8 · Task runner & evaluator

In [9]:
"""
run_task()        – runs one factual task on a loaded Llama instance.
evaluate_result() – asks the evaluator model for a YES/NO verdict.

Both functions use create_chat_completion() directly on the Llama object.
Token counts are read from response["usage"].

Evaluator bug fixed: the original code used max_tokens=4 and checked
  verdict.strip().upper().startswith("YES")
Small models (e.g. 1.2B) commonly emit leading whitespace, lowercase,
punctuation, or BOS tokens before the word, causing all verdicts to be
False.  Fix: use response_format={"type": "json_object"} together with
a structured prompt so the model is constrained to emit valid JSON with
a "correct" boolean field.  This removes all fragile string-parsing.
"""

_SYSTEM_PROMPT = "Answer the question you are given and justify your answer."

_EVALUATOR_SYSTEM_PROMPT: str = (
    "You are an answer evaluator. "
    "Given a question, an expected answer, and a model's answer, "
    "determine whether the model's answer is correct, even if it is "
    "more verbose, differently phrased, formatted with markdown/LaTeX, "
    "or contains additional accurate context. "
    "Treat answers as correct if the core factual content matches the "
    "expected answer semantically. "
    "Ignore harmless differences in wording, formatting, punctuation, "
    "capitalization, articles, explanations, or surrounding context. "
    "Accept concise and expanded answers equally, as long as they are "
    "factually consistent with the expected answer and do not introduce "
    "material inaccuracies or contradictions. "
    'Respond with ONLY valid JSON in this exact form: {"correct": true} or {"correct": false}. '
    "Output no other text, no markdown fences, no explanation."
)

# JSON schema that constrains the evaluator to {"correct": <bool>}.
# llama-cpp-python accepts this in the response_format field and uses
# its built-in grammar engine to enforce it at the token level.
_VERDICT_SCHEMA: dict = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {"correct": {"type": "boolean"}},
        "required": ["correct"],
        "additionalProperties": False,
    },
}


def run_task(llm: Llama, task: BenchmarkTask, model_name: str) -> TaskResult:
    """Execute one benchmark task on a loaded model and capture all metrics.

    Args:
        llm:        Loaded :class:`llama_cpp.Llama` instance.
        task:       The :class:`BenchmarkTask` to execute.
        model_name: Display name stored in the returned record.

    Returns:
        A :class:`TaskResult` with the model answer and performance metrics.
        On failure, ``error`` is set and numeric fields default to ``0``.
    """
    start = time.perf_counter()
    try:
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": _SYSTEM_PROMPT},
                {"role": "user",   "content": task.prompt},
            ],
            max_tokens=1024,
            temperature=0.0,
        )
        elapsed = time.perf_counter() - start

        usage         = response.get("usage", {})
        output_tokens = usage.get("completion_tokens", 0) or 0
        input_tokens  = usage.get("prompt_tokens", 0)    or 0
        answer        = response["choices"][0]["message"]["content"] or ""

        return TaskResult(
            model=model_name,
            task_id=task.task_id,
            prompt=task.prompt,
            expected_answer=task.expected_answer,
            model_answer=answer,
            elapsed_seconds=elapsed,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            tokens_per_second=output_tokens / elapsed if elapsed > 0 else 0.0,
            response_length=len(answer),
        )
    except Exception as exc:
        elapsed = time.perf_counter() - start
        return TaskResult(
            model=model_name,
            task_id=task.task_id,
            prompt=task.prompt,
            expected_answer=task.expected_answer,
            model_answer="",
            elapsed_seconds=elapsed,
            input_tokens=0,
            output_tokens=0,
            tokens_per_second=0.0,
            response_length=0,
            error=str(exc),
        )


def evaluate_result(evaluator_llm: Llama, result: TaskResult) -> bool:
    """Ask the evaluator model whether a TaskResult's answer is correct.

    Uses response_format with a JSON schema so llama-cpp-python's built-in
    grammar engine constrains the output to {"correct": <bool>}.  This
    eliminates the fragile YES/NO text-parsing that caused all correctly
    answered questions to be marked wrong (small models would prepend
    whitespace, emit lowercase, add punctuation, etc.).

    Args:
        evaluator_llm: Loaded Llama instance for the evaluator model.
        result:        The :class:`TaskResult` to judge.

    Returns:
        ``True`` when the evaluator deems the answer correct, else ``False``.
    """
    if result.error or not result.model_answer.strip():
        return False

    prompt = (
        f"Question: {result.prompt}\n"
        f"Expected answer: {result.expected_answer}\n"
        f"Model answer: {result.model_answer}\n"
        "Is the model's answer correct? Respond with {\"correct\": true} or {\"correct\": false}."
    )

    try:
        response = evaluator_llm.create_chat_completion(
            messages=[
                {"role": "system", "content": _EVALUATOR_SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            response_format=_VERDICT_SCHEMA,
            max_tokens=16,
            temperature=0.0,
        )
        raw = response["choices"][0]["message"]["content"] or ""
        verdict = json.loads(raw)
        return bool(verdict.get("correct", False))
    except Exception:
        return False


## 9 · Tool-calling runner

In [10]:
"""
run_tool_benchmark() tests each tool in isolation.

Each tool is tested with an independent create_chat_completion() call so
results are fully isolated.  Three fixes are applied vs. the original:

FIX 1 – Send only ONE tool per request instead of all 10.
  Sending all 10 tool definitions to a sub-2B parameter model overwhelms its
  instruction-following capacity; the model sees too many choices and falls
  back to generating plain text.  Isolating the relevant tool removes this
  ambiguity entirely.

FIX 2 – Use tool_choice={"type": "function", "function": {"name": tool_name}}
  instead of tool_choice="auto".
  With "auto" the model is free to answer in plain text, which sub-2B models
  almost always do.  Pinning the tool_choice to the specific function forces
  the chatml-function-calling handler to emit a tool_call block unconditionally.
  This is the standard way to test whether a model *can* call a tool at all,
  as opposed to testing whether it *chooses* to.

FIX 3 – Increase max_tokens from 256 → 512.
  The chatml-function-calling prompt wrapper adds ~50-80 tokens of overhead
  (system tokens, schema injection, separator tokens).  Combined with the JSON
  arguments the model must emit, 256 tokens was sometimes too tight, causing
  truncated or missing tool_call blocks.
"""

_TOOL_PROMPTS: dict[str, str] = {
    "get_current_date":      "What is today\'s date? Use a tool if you have one.",
    "get_tasks":             "What tasks do I have pending? Use a tool if you have one.",
    "list_files":            "List the files in the current directory using a tool.",
    "get_weather":           "What\'s the weather like in Paris right now? Use a tool.",
    "calculator":            "What is (17 + 5) * 3? Use the calculator tool.",
    "get_user_profile":      "Who am I logged in as? Use a tool to fetch my profile.",
    "search_knowledge_base": "Search the knowledge base for information about VPN setup.",
    "create_reminder":       "Set a reminder for me to call the dentist on 2025-09-01.",
    "get_stock_price":       "What is the current stock price of AAPL? Use a tool.",
    "translate_text":        "Translate \'Good morning, how are you?\' into French using a tool.",
}

# Lookup: tool_name → single tool definition dict (built once from TOOL_DEFINITIONS)
_TOOL_DEF_MAP: dict[str, dict] = {
    t["function"]["name"]: t for t in TOOL_DEFINITIONS
}


def _run_single_tool(llm: Llama, tool_name: str, user_prompt: str) -> ToolResult:
    """Ask the model to call *tool_name* and validate the result.

    Args:
        llm:         Loaded Llama instance.
        tool_name:   Expected tool the model should call.
        user_prompt: Natural-language prompt that motivates the tool call.

    Returns:
        A :class:`ToolResult` describing what happened.
    """
    # FIX 1: expose only the one tool under test so small models cannot get
    # confused by the 9 others and fall back to plain-text generation.
    single_tool_def = [_TOOL_DEF_MAP[tool_name]]

    # FIX 2: pin tool_choice to the exact function name so the chatml handler
    # is forced to emit a tool_call block rather than a text answer.
    forced_tool_choice = {"type": "function", "function": {"name": tool_name}}

    t0 = time.perf_counter()  # start timer BEFORE the completion call
    try:
        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a helpful assistant with access to tools. "
                        "When the user asks something a tool can answer, "
                        "call that tool immediately without narrating the plan."
                    ),
                },
                {"role": "user", "content": user_prompt},
            ],
            tools=single_tool_def,           # FIX 1: one tool only
            tool_choice=forced_tool_choice,  # FIX 2: force the call
            max_tokens=512,                  # FIX 3: more headroom for JSON args
            temperature=0.0,
        )
        elapsed = time.perf_counter() - t0  # capture AFTER call returns
        # clear_context(llm)  
        msg        = response["choices"][0]["message"]
        tool_calls = msg.get("tool_calls") or []

        if not tool_calls:
            return ToolResult(
                tool_name=tool_name,
                called=False,
                valid=False,
                raw="(no tool call emitted)",
                elapsed_seconds=elapsed,
            )

        call    = tool_calls[0]
        fn_name = call["function"]["name"]
        fn_args = call["function"]["arguments"]

        # Parse the arguments JSON and execute the static handler
        try:
            args_dict = json.loads(fn_args) if fn_args else {}
        except json.JSONDecodeError:
            args_dict = {}

        handler = TOOL_HANDLERS.get(fn_name)
        if handler is None:
            return ToolResult(
                tool_name=tool_name,
                called=True,
                valid=False,
                raw=f"Unknown tool called: {fn_name}",
                elapsed_seconds=elapsed,
            )

        result_dict = handler(args_dict)

        # Validate the tool result against its Pydantic schema
        validator = TOOL_RESULT_VALIDATORS[fn_name]
        validated = validator(**result_dict)

        return ToolResult(
            tool_name=tool_name,
            called=True,
            valid=True,
            raw=str(validated.model_dump()),
            elapsed_seconds=elapsed,
        )

    except Exception as exc:
        elapsed = time.perf_counter() - t0
        return ToolResult(
            tool_name=tool_name,
            called=False,
            valid=False,
            raw=f"ERROR: {exc}",
            elapsed_seconds=elapsed,
        )


def run_tool_benchmark(llm: Llama, model_name: str) -> ModelToolScore:
    """Run all tool tests against a loaded model.

    Args:
        llm:        Loaded Llama instance.
        model_name: Display name used in the returned record.

    Returns:
        A :class:`ModelToolScore` with one :class:`ToolResult` per tool.
    """
    results: list[ToolResult] = []

    for tool_name, prompt in _TOOL_PROMPTS.items():
        prefix = f"    [{tool_name}]"
        print(prefix, end=" ", flush=True)
        tr = _run_single_tool(llm, tool_name, prompt)
        clear_context(llm)  # flush KV cache between every tool call
        status = "✓ valid" if tr.valid else ("~ called, bad result" if tr.called else "✗ not called")
        print(f"{status}  ({tr.elapsed_seconds:.2f}s)  |  {tr.raw[:60]}")
        results.append(tr)

    return ModelToolScore(model=model_name, results=results)

## 10 · Main benchmark orchestrator

In [11]:
"""
Top-level orchestrator.

For each model in BENCHMARK_MODELS:
  1. Load model (Llama object) – passes chat_format from MODEL_CHAT_FORMATS
     so the correct completion handler is selected at construction time.
  2. Run factual benchmark tasks (sequential, one at a time).
     Factual tasks do NOT use tools, so chat_format does not matter here.
  3. Run tool-calling tests – requires chat_format to be set correctly;
     the handler selected at load time determines whether tool_calls can
     ever be non-empty.
  4. Unload model (free memory).

The evaluator model is loaded once after all candidate models have been
benchmarked.  It only needs to produce {correct: bool} JSON, so the
default chat_format (autodetect) is fine for it.

Sequential execution is intentional:
  • llama.cpp runs on the GPU/CPU serially anyway.
  • Avoids context-switching that would distort timing metrics.
  • Keeps progress logs readable.
"""


def run_benchmark(
    models: dict[str, str],
    tasks: list[BenchmarkTask],
) -> tuple[list[TaskResult], list[ModelToolScore]]:
    """Benchmark every model in *models* over *tasks* plus the tool suite.

    Args:
        models: Dict of display_name → .gguf path (from BENCHMARK_MODELS).
        tasks:  List of :class:`BenchmarkTask` to run.

    Returns:
        Tuple of:
          - flat list of :class:`TaskResult` (one per model×task)
          - list of :class:`ModelToolScore` (one per model)
    """
    all_results:     list[TaskResult]     = []
    all_tool_scores: list[ModelToolScore] = []
    pending_results: list[TaskResult]     = []  # awaiting evaluator

    # ── Phase 1: benchmark + tool tests for each model ───────────────────
    for model_name, model_path in models.items():
        sep = "=" * 66
        print()
        print(sep)
        print(f"  Model: {model_name}")
        print(sep)

        # 1. Load – pass the model-specific chat_format so tool calling works
        chat_fmt = MODEL_CHAT_FORMATS.get(model_name)  # None = autodetect
        llm = load_model(model_path, chat_format=chat_fmt)

        # 2. Factual benchmark
        n_t = len(tasks)
        print(f"  ── Factual tasks ({n_t}) ──────────────────────────────────")
        for task in tasks:
            prefix = f"  [{task.task_id:02d}] {task.prompt[:52]:<52}"
            print(prefix, end=" ", flush=True)

            result = run_task(llm, task, model_name)
            # clear_context(llm)
            if result.error:
                print(f"ERROR  {result.error[:60]}")
            else:
                print(
                    f"{result.elapsed_seconds:5.2f}s  "
                    f"{result.tokens_per_second:6.1f} tok/s  "
                    f"{result.response_length:4d} chars"
                )

            pending_results.append(result)
            all_results.append(result)

        # 3. Tool-calling tests
        print("  ── Tool-calling tests ───────────────────────────────────────────")
        tool_score = run_tool_benchmark(llm, model_name)
        all_tool_scores.append(tool_score)
        n_r = len(tool_score.results)
        print(f"  Tool score: {tool_score.score} / {n_r}")

        # 4. Unload — single clean path, no dangerous backend cycling
        print()
        unload_model(llm)

    # ── Phase 2: evaluate factual answers with the evaluator model ────────
    sep2 = "=" * 66
    print()
    print(sep2)
    print("  Evaluating factual answers …")
    print(sep2)

    evaluator_path = models[EVALUATOR_MODEL]
    print(f"  Loading evaluator: {EVALUATOR_MODEL}")
    # Evaluator only needs JSON output, not tool calling – autodetect is fine
    evaluator_llm = load_model(evaluator_path)

    for result in pending_results:
        result.is_correct = evaluate_result(evaluator_llm, result)
        verdict = "✓" if result.is_correct else "✗"
        label   = f"[{result.model}] T{result.task_id:02d}"
        print(f"  {label:<28} {verdict}")

    unload_model(evaluator_llm)

    print()
    print("Benchmark complete.")
    return all_results, all_tool_scores


# ── Run ───────────────────────────────────────────────────────────────────────
raw_results, tool_scores = run_benchmark(BENCHMARK_MODELS, TASKS)



  Model: Phi4-mini
  Loading ./models/phi-4-mini-instruct-q4_0.gguf … 

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


done (3.7s) [chatml-function-calling]
  ── Factual tasks (10) ──────────────────────────────────
  [01] What is the capital of France?                        1.99s    40.7 tok/s   356 chars
  [02] What is 2 raised to the power of 10?                  2.53s    45.5 tok/s   349 chars
  [03] What is the chemical symbol for gold?                 1.20s    45.0 tok/s   223 chars
  [04] Who wrote Romeo and Juliet?                           2.03s    44.3 tok/s   488 chars
  [05] What planet is closest to the Sun?                   24.97s    41.0 tok/s  5342 chars
  [06] In what year did World War II end?                    2.57s    38.9 tok/s   465 chars
  [07] What is the square root of 144?                       1.77s    39.0 tok/s   263 chars
  [08] How many sides does a hexagon have?                   2.79s    38.3 tok/s   419 chars
  [09] What is the boiling point of water in Celsius at sea  2.51s    36.7 tok/s   403 chars
  [10] What programming language is named after a British c  4.56s

KeyboardInterrupt: 

## 11 · Build results DataFrame

In [ ]:
"""
Flatten the list of TaskResult objects into a tidy pandas DataFrame.
One row per (model, task) pair with every metric as a column.

Bug fixed: previously `bool(r.is_correct)` silently converted None→False
for any result that had not yet been evaluated (e.g. if the evaluator
phase was skipped).  Now we use an explicit guard so unevaluated rows
show None rather than a misleading False.
"""


def build_dataframe(results: list[TaskResult]) -> pd.DataFrame:
    """Convert a list of :class:`TaskResult` objects into a tidy DataFrame.

    Args:
        results: Raw benchmark results from :func:`run_benchmark`.

    Returns:
        A :class:`pandas.DataFrame` with one row per (model, task) pair.
    """
    rows = [
        {
            "model":             r.model,
            "task_id":           r.task_id,
            "prompt":            r.prompt,
            "expected":          r.expected_answer,
            "answer":            r.model_answer,
            "elapsed_s":         r.elapsed_seconds,
            "input_tokens":      r.input_tokens,
            "output_tokens":     r.output_tokens,
            "tokens_per_second": r.tokens_per_second,
            "response_length":   r.response_length,
            # Guard against None: only evaluate_result() sets this field;
            # if it was never called (e.g. evaluator crashed) keep None
            # so the summary stats are not silently wrong.
            "is_correct":        bool(r.is_correct) if r.is_correct is not None else None,
            "error":             r.error,
        }
        for r in results
    ]
    return pd.DataFrame(rows)


df: pd.DataFrame = build_dataframe(raw_results)

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.2f}".format)
display(df[["model", "task_id", "expected", "answer",
            "elapsed_s", "tokens_per_second", "response_length", "is_correct"]])


## 12 · Per-model summary statistics

In [ ]:
"""
Aggregate the tidy DataFrame into one summary row per model.

Columns produced:
  avg_tps        – mean output tokens per second across all tasks.
  tasks_correct  – count of tasks judged correct by the evaluator.
  accuracy_pct   – tasks_correct / total_tasks * 100.
  avg_time_s     – mean wall-clock seconds per task.
  avg_length     – mean response length in characters.
  tool_score     – number of tool calls that were both emitted and valid (max 3).
"""
n_tasks: int = len(TASKS)

# Build a tool-score lookup: model → score
tool_score_map: dict[str, int] = {ts.model: ts.score for ts in tool_scores}

summary: pd.DataFrame = (
    df.groupby("model")
    .agg(
        avg_tps=("tokens_per_second", "mean"),
        tasks_correct=("is_correct", "sum"),
        avg_time_s=("elapsed_s", "mean"),
        avg_length=("response_length", "mean"),
    )
    .assign(
        accuracy_pct=lambda x: x["tasks_correct"] / n_tasks * 100,
        tool_score=lambda x: x.index.map(tool_score_map),
    )
    .reset_index()
    .sort_values("avg_tps", ascending=False)
)

display(summary)

## 13 · Tool-calling results table

In [ ]:
"""
Flatten the ModelToolScore list into a tidy DataFrame for display.
"""


def build_tool_dataframe(scores: list[ModelToolScore]) -> pd.DataFrame:
    rows = [
        {
            "model":     ts.model,
            "tool":      tr.tool_name,
            "called":    tr.called,
            "valid":     tr.valid,
            "result":    tr.raw[:80],
        }
        for ts in scores
        for tr in ts.results
    ]
    return pd.DataFrame(rows)


tool_df = build_tool_dataframe(tool_scores)

# Summary pivot: model × tool → valid?
pivot = tool_df.pivot_table(
    index="model", columns="tool", values="valid", aggfunc="first"
)
print("Tool call validity (True = called + valid result):")
display(pivot)

print()
print("Detailed tool call log:")
display(tool_df)

## 14 · Visualisation dashboard

In [ ]:
"""
Render a 2 × 3 dashboard of the five key benchmark metrics.

Panel layout
────────────
  [0,0] Throughput   – avg output tokens / second (higher = faster).
  [0,1] Accuracy     – tasks judged correct (higher = smarter).
  [0,2] Tool score   – tool calls that succeeded out of 3 (higher = better).
  [1,0] Latency      – avg wall-clock seconds / task (lower = faster).
  [1,1] Conciseness  – avg response length in chars (lower = more concise).
  [1,2] (empty / reserved)

The figure is saved as benchmark_results.png alongside the notebook.
"""


def plot_benchmark(summary: pd.DataFrame, n_tasks: int) -> None:
    """Render a 2×3 metric dashboard from the per-model summary DataFrame."""
    models: list[str] = summary["model"].tolist()
    x: np.ndarray     = np.arange(len(models))
    palette            = plt.cm.tab10(np.linspace(0, 0.9, len(models)))

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    fig.suptitle(
        "AI Model Benchmark Dashboard (llama-cpp-python)",
        fontsize=18, fontweight="bold", y=1.02,
    )

    def _style_axis(ax, title, ylabel, bars, fmt, ylim_top=None):
        ax.set_title(title, fontweight="bold", fontsize=11, pad=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=28, ha="right", fontsize=8)
        ax.bar_label(bars, fmt=fmt, padding=4, fontsize=9, fontweight="bold")
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.grid(axis="y", alpha=0.3, linewidth=0.8)
        ax.spines[["top", "right"]].set_visible(False)
        if ylim_top is not None:
            ax.set_ylim(0, ylim_top)

    # [0,0] Throughput
    ax = axes[0, 0]
    bars = ax.bar(x, summary["avg_tps"], color=palette, edgecolor="white", linewidth=0.6)
    _style_axis(ax, "Throughput (avg tokens / s)", "Tokens / second", bars, "%.1f")

    # [0,1] Accuracy
    ax = axes[0, 1]
    bars = ax.bar(x, summary["tasks_correct"], color=palette, edgecolor="white", linewidth=0.6)
    _style_axis(
        ax, f"Accuracy (correct / {n_tasks} tasks)", "Correct answers",
        bars, "%d", ylim_top=n_tasks + 1.5,
    )
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # [0,2] Tool score
    ax = axes[1,1]
    bars = ax.bar(x, summary["tool_score"], color=palette, edgecolor="white", linewidth=0.6)
    _style_axis(
        ax, "Tool score (valid calls / 10)", "Valid tool calls",
        bars, "%d", ylim_top=10+ 1.5,
    )
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # [1,0] Latency
    ax = axes[1, 0]
    bars = ax.bar(x, summary["avg_time_s"], color=palette, edgecolor="white", linewidth=0.6)
    _style_axis(ax, "Latency (avg seconds / task)", "Seconds", bars, "%.2f")

    fig.tight_layout()
    output_path = "benchmark_results.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Dashboard saved → {output_path}")


plot_benchmark(summary, n_tasks)

## 15 · Per-task accuracy heatmap

In [ ]:
"""
Heatmap showing which (model, task) pairs were answered correctly.
Rows are models; columns are task IDs.
Green = correct, Red = wrong.
"""


def plot_heatmap(df: pd.DataFrame) -> None:
    """Render a correctness heatmap over the (model × task) space."""
    pivot_correct: pd.DataFrame = df.pivot_table(
        index="model", columns="task_id", values="is_correct", aggfunc="first",
    ).astype(float)

    pivot_time: pd.DataFrame = df.pivot_table(
        index="model", columns="task_id", values="elapsed_s", aggfunc="first",
    )

    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot_correct) * 0.9)))

    im = ax.imshow(pivot_correct.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(pivot_correct.columns)))
    ax.set_xticklabels([f"T{c}" for c in pivot_correct.columns], fontsize=10)
    ax.set_yticks(range(len(pivot_correct.index)))
    ax.set_yticklabels(pivot_correct.index, fontsize=9)
    ax.set_xlabel("Task ID", fontsize=11)
    ax.set_ylabel("Model", fontsize=11)
    ax.set_title(
        "Per-task correctness heatmap  (green = correct, red = wrong)",
        fontsize=13, fontweight="bold", pad=12,
    )

    for row_idx, model in enumerate(pivot_correct.index):
        for col_idx, task_id in enumerate(pivot_correct.columns):
            elapsed = pivot_time.at[model, task_id]
            label = f"{elapsed:.2f}s" if pd.notna(elapsed) else "—"
            ax.text(col_idx, row_idx, label, ha="center", va="center",
                    fontsize=9, fontweight="bold", color="black")

    task_legend = "  ·  ".join(f"T{t.task_id}: {t.prompt[:30]}" for t in TASKS)
    fig.text(0.5, -0.05, task_legend, ha="center", fontsize=7, wrap=True, color="#444")

    fig.colorbar(im, ax=ax, shrink=0.6, label="Correct (1) / Wrong (0)")
    fig.tight_layout()
    output_path = "benchmark_heatmap.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Heatmap saved → {output_path}")


plot_heatmap(df)

## 16 · Tool-calling heatmap

In [ ]:
"""
Heatmap showing per-model, per-tool success.
Green  = tool was called and result validated correctly.
Yellow = tool was called but result failed validation.
Red    = tool was not called at all.

Note: ToolResult is defined once in cell 4 (Data models). The duplicate
definition that was previously here has been removed to avoid silently
shadowing the original class when cells are re-run out of order.
"""
def plot_tool_heatmap(scores: list[ModelToolScore]) -> None:
    """Render a tool-call success heatmap with per-cell elapsed time.

    Args:
        scores: List of :class:`ModelToolScore`, one per benchmarked model.
    """
    tool_names  = [t["function"]["name"] for t in TOOL_DEFINITIONS]
    model_names = [ts.model for ts in scores]

    matrix      = np.zeros((len(model_names), len(tool_names)))
    time_matrix = [[None for _ in tool_names] for _ in model_names]

    for r_idx, ts in enumerate(scores):
        result_map = {tr.tool_name: tr for tr in ts.results}
        for c_idx, tool_name in enumerate(tool_names):
            tr = result_map.get(tool_name)
            if tr is None:
                matrix[r_idx, c_idx] = 0.0
            elif tr.valid:
                matrix[r_idx, c_idx] = 1.0
            elif tr.called:
                matrix[r_idx, c_idx] = 0.5
            else:
                matrix[r_idx, c_idx] = 0.0
            time_matrix[r_idx][c_idx] = tr.elapsed_seconds if tr is not None else None

    fig, ax = plt.subplots(figsize=(10, max(3, len(model_names) * 0.9)))
    im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(tool_names)))
    ax.set_xticklabels(tool_names, fontsize=10, rotation=20, ha="right")
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names, fontsize=9)
    ax.set_title(
        "Tool-calling heatmap  (✓ valid · ~ called/bad · ✗ not called)",
        fontsize=13, fontweight="bold", pad=12,
    )

    for r_idx in range(len(model_names)):
        for c_idx in range(len(tool_names)):
            elapsed = time_matrix[r_idx][c_idx]
            label = f"{elapsed:.2f}s" if elapsed is not None else "—"
            ax.text(c_idx, r_idx, label,
                    ha="center", va="center", fontsize=9,
                    fontweight="bold", color="black")

    fig.colorbar(im, ax=ax, shrink=0.6,
                 label="0=not called · 0.5=called/invalid · 1=valid")
    fig.tight_layout()
    output_path = "tool_heatmap.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Tool heatmap saved → {output_path}")

    
plot_tool_heatmap(tool_scores)


## 17 · System information

In [ ]:
import platform
import subprocess

OUTPUT_FILE = "system_info.txt"


def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        return f"Command failed: {e}"


def parse_nvidia_smi(output):
    lines  = output.strip().splitlines()
    parsed = []
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) == 6:
            parsed.append({
                "Name":                 parts[0],
                "Driver Version":       parts[1],
                "Total Memory (MiB)":   parts[2],
                "Used Memory (MiB)":    parts[3],
                "Free Memory (MiB)":    parts[4],
                "GPU Temperature (°C)": parts[5],
            })
    return parsed


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    def write_line(text=""):
        print(text)
        f.write(text + "\n")

    write_line("=== SYSTEM INFORMATION ===")
    write_line(f"Machine:   {platform.machine()}")
    write_line(f"Version:   {platform.version()}")
    write_line(f"Platform:  {platform.platform()}")
    write_line(f"System:    {platform.system()}")
    write_line(f"Processor: {platform.processor()}")

    write_line("\n=== GPU INFORMATION ===")
    raw = run_cmd([
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,memory.used,memory.free,temperature.gpu",
        "--format=csv,noheader",
    ])

    if "command failed" in raw.lower():
        write_line("nvidia-smi not available or GPU not detected.")
    else:
        gpus = parse_nvidia_smi(raw)
        if not gpus:
            write_line("No GPU data found.")
        else:
            for i, gpu in enumerate(gpus):
                write_line(f"\n--- GPU {i} ---")
                for key, value in gpu.items():
                    write_line(f"{key}: {value}")

print(f"\nSaved system info to: {OUTPUT_FILE}")